# 구매 기록 테이블 전처리 수행

In [22]:
import pandas as pd

In [3]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [4]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_paymenthistory`
"""

# 판다스 데이터프레임으로 변환|
df = client.query(sql).to_dataframe()

print(df.head())

      id  productId phone_type                created_at  user_id
0  89584  heart.777          A 2023-06-06 04:58:49+00:00   835888
1  89585  heart.200          A 2023-06-06 04:59:22+00:00   835888
2   2403  heart.777          A 2023-05-14 04:22:44+00:00   837641
3  79774  heart.777          A 2023-05-29 10:13:55+00:00   837737
4    195  heart.777          A 2023-05-13 23:10:10+00:00   837842


## 결측치 확인 및 데이터 정보 확인

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95140 entries, 0 to 95139
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype              
---  ------      --------------  -----              
 0   id          95140 non-null  Int64              
 1   productId   95140 non-null  str                
 2   phone_type  95140 non-null  str                
 3   created_at  95140 non-null  datetime64[us, UTC]
 4   user_id     95140 non-null  Int64              
dtypes: Int64(2), datetime64[us, UTC](1), str(2)
memory usage: 4.7 MB


* 결측치(Missing Value) 없음

## 중복값 체크

In [ ]:
# 중복값 체크 
df.duplicated().sum()

df['id'].duplicated().sum()

df[df[['productId','phone_type','user_id','created_at']].duplicated(keep=False)]

df[df[['phone_type','user_id','created_at']].duplicated(keep=False)]

,id,productId,phone_type,created_at,user_id
1769,97428,heart.777,A,2023-11-30 16:23:06+00:00,928960
3137,97389,heart.1000,A,2023-11-19 09:52:36+00:00,974093
5265,97570,heart.777,A,2023-12-21 03:41:18+00:00,1044654
6107,97705,heart.200,A,2024-01-11 07:03:55+00:00,1067239
7010,97591,heart.777,A,2023-12-25 15:46:45+00:00,1091156
...,...,...,...,...,...
95113,97449,heart.1000,I,2023-12-05 09:13:43+00:00,1579025
95121,94540,heart.200,I,2023-07-26 11:20:13+00:00,1579249
95131,96958,heart.777,I,2023-09-27 10:41:13+00:00,1580828
95133,96960,heart.777,I,2023-09-27 10:41:31+00:00,1580828


* 결제 데이터 중복 적재 이상 식별:

  * 현상 분석: 동일 유저(user_id) 기준으로 1초의 오차도 없이 동일 시각에 복수의 구매 이력이 기록된 사례가 다수 확인됨.

  * 도메인/비즈니스 검증: 서비스 기획 구조상 장바구니 일괄 결제 기능이 부재하며 단일 건 구매만 지원하므로, 동일 시점에 복수 결제가 일어나는 것은 물리적으로 불가능함.

  * 원인 확정: 네트워크 타임아웃, 클라이언트 다중 클릭(더블 클릭), 또는 결제 웹훅(Webhook) 처리 파이프라인의 일시적 오류로 인한 중복 적재로 판단.

* 정제(Deduplication) 조치:

  * 중복 결제 트랜잭션 중 발생 시각 기준 최초 1건(First Record)만 정상 결제로 인정하고, 이후 생성된 나머지 중복 레코드는 전량 삭제(또는 제외) 처리 결정.

## 이상치 확인

In [20]:
df['phone_type'].unique()

<ArrowStringArray>
['A', 'I']
Length: 2, dtype: str

In [23]:
# 1. 날짜 변환 시도 (문법/달력상 불가능한 날짜는 NaT로 변환됨)
converted = pd.to_datetime(df['created_at'], errors='coerce')

# 2. 원래 값이 있었는데(notna), 날짜 파싱에 실패해 NaT가 된 비정상 행 필터링
invalid_dates = df[df['created_at'].notna() & converted.isna()]

print(f"비상식적인 날짜 데이터 수: {len(invalid_dates)}건")
display(invalid_dates[['created_at']])

비상식적인 날짜 데이터 수: 0건


,created_at


In [ ]:
# 중복 데이터 531건 삭제 진행

delete_sql = f"""
DELETE FROM `{PROJECT_ID}.{DATA_SET}.accounts_paymenthistory`
WHERE id IN (
    SELECT id
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_paymenthistory`
    QUALIFY ROW_NUMBER() OVER(PARTITION BY user_id, created_at ORDER BY id ASC) > 1
);
"""

# 위 쿼리와 동일한 기능 (QUALIFY절은 빅쿼리를 포함한 일부 엔진에서만 지원)
# DELETE FROM accounts_paymenthistory
# WHERE id IN (
#     SELECT id 
#     FROM (SELECT *
#             , ROW_NUMBER() OVER(PARTITION BY user_id, created_at ORDER BY id ASC) AS RN
#         FROM accounts_paymenthistory
#         ) AS t
#     WHERE RN > 1
# )

query_job = client.query(delete_sql)
query_job.result()  # DML 완료 대기

print(f"\n삭제 완료! 총 삭제된 행 수: {query_job.num_dml_affected_rows}건")